In [6]:
import torch
import torchvision
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import os


In [7]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])


In [6]:
data_dir = r"C:\Users\91994\Desktop\acc_det\data"

train_data = datasets.ImageFolder(os.path.join(data_dir, 'train'), transform=transform)
val_data = datasets.ImageFolder(os.path.join(data_dir, 'val'), transform=transform)
test_data = datasets.ImageFolder(os.path.join(data_dir, 'test'), transform=transform)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

In [7]:
model = models.resnet18(pretrained=True)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)  # Two classes: Accident, Not Accident

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

C:\Users\91994\AppData\Roaming\Python\Python311\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [9]:
num_epochs = 10  # Change as needed
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}")

Epoch 1, Loss: 0.8024
Epoch 2, Loss: 0.5125
Epoch 3, Loss: 0.4173
Epoch 4, Loss: 0.3010
Epoch 5, Loss: 0.2275
Epoch 6, Loss: 0.1530
Epoch 7, Loss: 0.1250
Epoch 8, Loss: 0.1537
Epoch 9, Loss: 0.1361
Epoch 10, Loss: 0.1514


In [19]:
from PIL import Image

def predict(image_path, model):
    model.eval()
    image = Image.open(image_path)
    image = transform(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = model(image)
        _, predicted = torch.max(output, 1)
    
    return "Accident" if predicted.item() == 0 else "Not Accident"

print(predict(r"C:\Users\91994\Desktop\acc_det\data\test\Non Accident\test17_12.jpg", model))


Not Accident


In [11]:
save_path = r"C:\Users\91994\Desktop\acc_det\data\accident_detection.pth"
torch.save(model.state_dict(), save_path)
print(f"Model saved at: {save_path}")

Model saved at: C:\Users\91994\Desktop\acc_det\data\accident_detection.pth


In [ ]:
print(device)

cpu


In [10]:
import torch
import torch.nn as nn
import torchvision.models as models

# Define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the same architecture
model = models.resnet18(pretrained=False)  # Don't load default weights
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)  # Two classes: Accident, Not Accident

# Load state_dict
model_path = r"C:\Users\91994\Desktop\acc_det\data\accident_detection.pth"
state_dict = torch.load(model_path, map_location=device)  # Load weights

# Load the weights into the model
model.load_state_dict(state_dict)

# Move model to the correct device
model = model.to(device)

# Set model to evaluation mode
model.eval()

print("✅ Model loaded successfully!")


✅ Model loaded successfully!


C:\Users\91994\AppData\Local\Temp\ipykernel_8540\1030883436.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(model_path, map_location=device)  # 

In [20]:
from flask import Flask, request, jsonify, render_template, send_from_directory
from flask_cors import CORS
import cv2
import numpy as np
import cvzone
from ultralytics import YOLO
import pytesseract
import os

# Configure Tesseract OCR (Change path if needed)
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

# Flask App
app = Flask(__name__, template_folder="templates", static_folder="static")
CORS(app)

# Load YOLO Model
MODEL_PATH = "Weights/best.pt"
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"YOLO model not found at {MODEL_PATH}")

yolo_model = YOLO(MODEL_PATH)
class_labels = ['With Helmet', 'Without Helmet']

# Directories
UPLOAD_FOLDER = "static/uploads"
PROCESSED_FOLDER = "static/processed"
CROPPED_FOLDER = "static/cropped"
os.makedirs(UPLOAD_FOLDER, exist_ok=True)
os.makedirs(PROCESSED_FOLDER, exist_ok=True)
os.makedirs(CROPPED_FOLDER, exist_ok=True)

# Dummy Vehicle Database
vehicle_db = {
    "AP16AB1234": {"name": "Rajesh Kumar", "vehicle": "Honda Activa"},
    "TS08CD5678": {"name": "Pooja Sharma", "vehicle": "Yamaha R15"},
}

def extract_plate(image_path):
    """ Extracts license plate number using OCR. """
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    plate_text = pytesseract.image_to_string(gray, config='--psm 7')
    return plate_text.strip().replace(" ", "").upper()

def check_plate(plate_number):
    """ Checks if the plate number exists in the database. """
    return vehicle_db.get(plate_number, None)

@app.route('/')
def home():
    return render_template('index.html')

@app.route('/detect', methods=['POST'])
def detect_helmet():
    """ Detects helmets and extracts license plates if necessary. """
    if 'image' not in request.files:
        return jsonify({"error": "No image uploaded"}), 400

    file = request.files['image']
    filename = file.filename
    input_path = os.path.join(UPLOAD_FOLDER, filename)
    output_path = os.path.join(PROCESSED_FOLDER, filename)
    cropped_path = os.path.join(CROPPED_FOLDER, filename)

    file.save(input_path)
    print(f"✅ Image Received: {input_path}")

    img = cv2.imread(input_path)
    if img is None:
        return jsonify({"error": "Invalid image format"}), 400

    h, w = img.shape[:2]  # Image dimensions

    print("🔍 Performing Helmet Detection...")
    results = yolo_model(img)
    
    without_helmet_boxes = []

    for r in results:
        boxes = r.boxes
        for box in boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])  # Bounding box
            conf = float(box.conf[0])
            cls = int(box.cls[0])  # Class index

            if conf > 0.3:
                label = class_labels[cls]
                color = (0, 255, 0) if cls == 0 else (0, 0, 255)  # Green (Helmet), Red (No Helmet)
                
                # Draw detection box
                cvzone.cornerRect(img, (x1, y1, x2-x1, y2-y1), colorC=color)  
                cvzone.putTextRect(img, f'{label} {conf:.2f}', (x1, y1 - 10), scale=0.8, thickness=1, colorR=(255, 0, 0))
                print(f"🟢 Detected: {label} | Confidence: {conf:.2f}")

                # If "Without Helmet", save bounding box
                if cls == 1:
                    without_helmet_boxes.append((x1, y1, x2, y2))

    if without_helmet_boxes:
        # Get leftmost and rightmost X coordinates
        min_x = min(box[0] for box in without_helmet_boxes) - 50  # Extend left
        max_x = max(box[2] for box in without_helmet_boxes) + 50  # Extend right

        # Ensure valid crop boundaries
        min_x = max(0, min_x)
        max_x = min(w, max_x)

        # Crop from top (0) to bottom of the image
        cropped_img = img[0:h, min_x:max_x]
        cv2.imwrite(cropped_path, cropped_img)
        print(f"🚀 Cropped Image Saved: {cropped_path}")

        # 🔍 Extract License Plate
        plate_number = extract_plate(cropped_path)
        print(f"🔡 Extracted Plate: {plate_number}")

        # 📜 Check in Database
        if plate_number:
            owner_details = check_plate(plate_number)
            if owner_details:
                print(f"🚔 Violator Found: {owner_details}")
                return jsonify({
                    "processed_image_url": f"/static/processed/{filename}",
                    "cropped_image_url": f"/static/cropped/{filename}",
                    "plate_number": plate_number,
                    "e_challan": f"E-Challan generated for {owner_details['name']} (Vehicle: {owner_details['vehicle']})"
                }), 200
            else:
                print("❌ No Matching Vehicle Found in Database.")
                return jsonify({
                    "processed_image_url": f"/static/processed/{filename}",
                    "cropped_image_url": f"/static/cropped/{filename}",
                    "plate_number": plate_number,
                    "e_challan": "No matching vehicle found."
                }), 200

    cv2.imwrite(output_path, img)
    print(f"✅ Processed Image Saved: {output_path}")

    return jsonify({
        "processed_image_url": f"/static/processed/{filename}"
    }), 200

@app.route('/static/processed/<filename>')
def serve_processed_image(filename):
    return send_from_directory(PROCESSED_FOLDER, filename)

@app.route('/static/cropped/<filename>')
def serve_cropped_image(filename):
    return send_from_directory(CROPPED_FOLDER, filename)

if __name__ == '__main__':
    app.run(debug=True, host='0.0.0.0', port=5000)


FileNotFoundError: YOLO model not found at Weights/best.pt